# COSMoS_BC_DSS_Validate -- QC Report for BC and Dataset Specializations

**Purpose:** Read the interim `COSMoS_BC_DSS.xlsx` produced by `COSMoS_BC_DSS_Flatten`
and run structural and curation-principle quality checks (QC-01 to QC-14).

**Input:** `interim/COSMoS_BC_DSS.xlsx`

**Output:** `reports/COSMoS_BC_DSS_QC.xlsx`

**Pipeline:**
- `COSMoS_BC_DSS_Flatten.ipynb` -- produces the interim file consumed here
- `COSMoS_BC_DSS_Validate.ipynb` (this notebook) -- independent QC pass

Validate can be re-run at any time without re-downloading or re-processing
COSMoS sources. All required data is read from the interim file.

## Quality Checks

Checks QC-01 through QC-10 are structural pipeline checks.
Checks QC-11 through QC-14 validate against the
[BC Curation Principles and Completion Guidelines](https://cdisc-org.github.io/COSMoS/bc_starter_package/doc/BC%20Curation%20Principles%20and%20Completion%20GLs.xlsx)
published in the COSMoS GitHub repository.

| # | Check | Severity |
|---|---|---|
| QC-01 | CT unmapped: Specimen values not resolved to CDISC CT | ERROR |
| QC-02 | CT unmapped: Method values not resolved | WARNING |
| QC-03 | CT unmapped: Unit values not resolved | WARNING |
| QC-04 | DSS rows with no TESTCD (no CT test code in COSMoS source) | INFO |
| QC-05 | DSS rows with Specimen but no Specimen_NCIt | WARNING |
| QC-06 | Quantitative DSS rows with no Allowed_Units | WARNING |
| QC-07 | Data corrections applied (expected -- documents known source issues) | INFO |
| QC-08 | Multi-result DSS: same TESTCD+Domain+Specimen, different DS_Code | INFO |
| QC-09 | BC_only_no_DS count by Category (gap analysis by editorial tag) | INFO |
| QC-10 | Retired BCs included in output (BC_Name contains RETIRED) | WARNING |
| QC-11a | Result_Scale='Qualitative' -- COSMoS source term, not in curation principles valid set | WARNING |
| QC-11b | Result_Scale='datetime' -- likely maps to Temporal per curation principles | WARNING |
| QC-11c | Result_Scale: unexpected value (not Qualitative or datetime) | ERROR |
| QC-12 | Placeholder BC_IDs (NEW_* prefix -- NCIt code not yet assigned) | INFO |
| QC-13 | BC_Synonyms: preferred term not stripped from synonym list | WARNING |
| QC-14 | TESTCD_NCIt differs from NCIt_Code (DSS test identity != BC identity) | INFO |

## 1. Configuration

In [ ]:
import pandas as pd
from pathlib import Path
import re
from collections import Counter
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter

# Directory structure - notebook is in notebooks/ subfolder
BASE_DIR    = Path.cwd().parent
INTERIM_DIR = BASE_DIR / 'interim'
REPORTS_DIR = BASE_DIR / 'reports'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INTERIM_FILE = INTERIM_DIR / 'COSMoS_BC_DSS.xlsx'
REPORT_FILE  = REPORTS_DIR / 'COSMoS_BC_DSS_QC.xlsx'

GENERATION_DATE = datetime.now().strftime('%Y-%m-%d')

assert INTERIM_FILE.exists(), f'Interim file not found: {INTERIM_FILE}\nRun COSMoS_BC_DSS_Flatten.ipynb first.'

print('Configuration:')
print(f'  Input:  {INTERIM_FILE}')
print(f'  Output: {REPORT_FILE}')

## 2. Load Interim File

In [ ]:
# Load primary data sheet
bc_dss_df = pd.read_excel(INTERIM_FILE, sheet_name='BC_DSS')

# Load CT unmapped log (built by Flatten, needed for QC-01/02/03)
ct_unmapped_df = pd.read_excel(INTERIM_FILE, sheet_name='CT_Unmapped_Log')

print(f'BC_DSS sheet:      {len(bc_dss_df)} rows x {len(bc_dss_df.columns)} columns')
print(f'CT_Unmapped_Log:   {len(ct_unmapped_df)} entries')
print(f'\nRow_Type distribution:')
for rt, cnt in bc_dss_df['Row_Type'].value_counts().items():
    print(f'  {rt:30} {cnt:>6}')

## 3. Reconstruct CT Unmapped Trackers

Reconstruct `UNMAPPED_SPECIMENS`, `UNMAPPED_METHODS`, `UNMAPPED_UNITS` dicts
from the `CT_Unmapped_Log` sheet serialized by Flatten.
These are consumed by QC-01, QC-02, and QC-03.

In [ ]:
# Reconstruct unmapped trackers: {value: ['x'] * usage_count}
# QC checks only need len() of the lists, so this is sufficient.
UNMAPPED_SPECIMENS = {}
UNMAPPED_METHODS   = {}
UNMAPPED_UNITS     = {}

for _, row in ct_unmapped_df.iterrows():
    typ   = str(row.get('Type', '')).strip()
    val   = str(row.get('Value', '')).strip()
    count = int(row.get('Usage_Count', 1))
    placeholder = ['x'] * count  # len() gives usage count for QC checks
    if typ == 'SPECIMEN':
        UNMAPPED_SPECIMENS[val] = placeholder
    elif typ == 'METHOD':
        UNMAPPED_METHODS[val] = placeholder
    elif typ == 'UNIT':
        UNMAPPED_UNITS[val] = placeholder

print(f'Reconstructed unmapped trackers:')
print(f'  UNMAPPED_SPECIMENS: {len(UNMAPPED_SPECIMENS)}')
print(f'  UNMAPPED_METHODS:   {len(UNMAPPED_METHODS)}')
print(f'  UNMAPPED_UNITS:     {len(UNMAPPED_UNITS)}')

## 4. Prepare QC Views

Derive the filtered views used across multiple QC checks.

In [ ]:
# DSS rows only (Row_Type == 'DSS') -- used by most checks
v = bc_dss_df[bc_dss_df['Row_Type'] == 'DSS'].copy()

# Ensure string columns are clean
for col in ['TESTCD', 'Specimen', 'Specimen_NCIt', 'Result_Scale',
            'Allowed_Units', 'BC_Name', 'COSMoS_BC_ID', 'BC_Synonyms',
            'DS_Code', 'Categories', 'Domain']:
    if col in bc_dss_df.columns:
        bc_dss_df[col] = bc_dss_df[col].fillna('').astype(str).str.strip()
        v[col]         = v[col].fillna('').astype(str).str.strip() if col in v.columns else v.get(col, '')

dss_rows = v  # alias used in stats-style checks

n_dss_rows   = (bc_dss_df['Row_Type'] == 'DSS').sum()
n_no_ds_rows = (bc_dss_df['Row_Type'] == 'BC_only_no_DS').sum()
n_hier_rows  = (bc_dss_df['Row_Type'] == 'BC_only_hierarchy').sum()

print(f'QC views prepared:')
print(f'  DSS rows (v):          {len(v)}')
print(f'  BC_only_no_DS rows:    {n_no_ds_rows}')
print(f'  BC_only_hierarchy rows:{n_hier_rows}')

## 5. Run Quality Checks (QC-01 to QC-14)

In [ ]:
qc_results = []   # list of dicts -- written to QC_Report sheet

def qc_pass(check_id, description, detail=''):
    qc_results.append({'Check': check_id, 'Severity': 'PASS', 'Description': description,
                        'Count': 0, 'Detail': detail})
    print(f'  [PASS] {check_id} {description}')

def qc_flag(check_id, severity, description, count, detail=''):
    qc_results.append({'Check': check_id, 'Severity': severity, 'Description': description,
                        'Count': count, 'Detail': detail})
    tag = f'[{severity}]'
    print(f'  {tag:9} {check_id} {description} -- {count} item(s)')
    if detail:
        for line in detail.split('|'):
            if line.strip():
                print(f'             {line.strip()}')

def qc_info(check_id, description, count, detail=''):
    qc_results.append({'Check': check_id, 'Severity': 'INFO', 'Description': description,
                        'Count': count, 'Detail': detail})
    print(f'  [INFO]    {check_id} {description} -- {count}')
    if detail:
        for line in detail.split('|'):
            if line.strip():
                print(f'             {line.strip()}')

v = bc_dss_df[bc_dss_df['Row_Type'] == 'DSS'].copy()

print('=' * 70)
print('QUALITY REPORT')
print('=' * 70)

# QC-01: Unmapped specimens
if UNMAPPED_SPECIMENS:
    detail = ' | '.join(f'{k} ({len(v2)} usages)' for k, v2 in sorted(UNMAPPED_SPECIMENS.items()))
    qc_flag('QC-01', 'ERROR', 'CT unmapped: Specimen', len(UNMAPPED_SPECIMENS), detail)
else:
    qc_pass('QC-01', 'CT unmapped: Specimen')

# QC-02: Unmapped methods
if UNMAPPED_METHODS:
    detail = ' | '.join(f'{k} ({len(v2)} usages)' for k, v2 in sorted(UNMAPPED_METHODS.items()))
    qc_flag('QC-02', 'WARNING', 'CT unmapped: Method', len(UNMAPPED_METHODS), detail)
else:
    qc_pass('QC-02', 'CT unmapped: Method')

# QC-03: Unmapped units
if UNMAPPED_UNITS:
    detail = ' | '.join(f'{k} ({len(v2)} usages)' for k, v2 in sorted(list(UNMAPPED_UNITS.items())[:15]))
    if len(UNMAPPED_UNITS) > 15:
        detail += f' | ... and {len(UNMAPPED_UNITS) - 15} more'
    qc_flag('QC-03', 'WARNING', 'CT unmapped: Unit', len(UNMAPPED_UNITS), detail)
else:
    qc_pass('QC-03', 'CT unmapped: Unit')

# QC-04: DS rows with blank TESTCD
blank_testcd = v[v['TESTCD'].fillna('').astype(str).str.strip().isin(['', 'nan'])]
if len(blank_testcd) > 0:
    detail = ' | '.join(f'{r["DS_Code"]} ({r["Domain"]})' for _, r in blank_testcd.head(10).iterrows())
    qc_flag('QC-04', 'INFO', 'DSS rows with no TESTCD (no CT test code in COSMoS source)', len(blank_testcd), detail)
else:
    qc_pass('QC-04', 'DSS rows with no TESTCD')

# QC-05: Specimen present but Specimen_NCIt blank (unmapped to CT)
spec_no_ncit = v[
    (v['Specimen'].astype(str).str.strip().isin(['', 'nan']) == False) &
    (v['Specimen_NCIt'].astype(str).str.strip().isin(['', 'nan']))
]
if len(spec_no_ncit) > 0:
    specimens = spec_no_ncit['Specimen'].unique()
    detail = ' | '.join(specimens[:10])
    qc_flag('QC-05', 'WARNING', 'Specimen present but Specimen_NCIt blank', len(spec_no_ncit), detail)
else:
    qc_pass('QC-05', 'Specimen present but Specimen_NCIt blank')

# QC-06: Quantitative DS rows with no Allowed_Units
qn_no_units = v[
    (v['Result_Scale'] == 'Quantitative') &
    (v['Allowed_Units'].astype(str).str.strip().isin(['', 'nan']))
]
if len(qn_no_units) > 0:
    by_dom = qn_no_units['Domain'].value_counts().to_dict()
    detail = ' | '.join(f'{d}: {n}' for d, n in sorted(by_dom.items()))
    qc_flag('QC-06', 'WARNING', 'Quantitative DS rows with no Allowed_Units', len(qn_no_units), detail)
else:
    qc_pass('QC-06', 'Quantitative DS rows with no Allowed_Units')

# QC-07: Known COSMoS source discrepancies (not corrected -- shown as-is)
# GLUCSERPL uses SERUM (C13325); all other *SERPL DS use SERUM OR PLASMA (C105706)
glucserpl_spec = v[v['DS_Code'] == 'GLUCSERPL']['Specimen'].values
glucserpl_val = glucserpl_spec[0] if len(glucserpl_spec) else 'not found'
qc_flag('QC-07', 'WARNING',
        'GLUCSERPL specimen is SERUM -- expected SERUM OR PLASMA per *SERPL pattern',
        1 if glucserpl_val == 'SERUM' else 0,
        f'GLUCSERPL Specimen={glucserpl_val!r} | '
        'Recommend reporting to CDISC COSMoS curators for correction at source')

# QC-08: Multi-result DS (same TESTCD+Domain+Specimen, different DS_Code)
# Expected pattern in GF and other genomics domains -- document, not an error
has_spec = v[v['Specimen'].astype(str).str.strip().isin(['', 'nan']) == False].copy()
multi = has_spec.groupby(['Domain', 'TESTCD', 'Specimen']).size().reset_index(name='count')
multi = multi[multi['count'] > 1].sort_values('count', ascending=False)
if len(multi) > 0:
    detail = ' | '.join(
        f'{r["Domain"]} {r["TESTCD"]} ({r["Specimen"]}) x{r["count"]}'
        for _, r in multi.head(10).iterrows()
    )
    qc_info('QC-08', 'Multi-result DSS (same TESTCD+Domain+Specimen)', len(multi), detail)
else:
    qc_info('QC-08', 'Multi-result DSS (same TESTCD+Domain+Specimen)', 0)

# QC-09: BC_only_no_DS by Category (gap analysis)
nods = bc_dss_df[bc_dss_df['Row_Type'] == 'BC_only_no_DS']
nods_tags = []
for cats_str in nods['Categories']:
    if cats_str and str(cats_str).strip():
        for tag in str(cats_str).split(';'):
            tag = tag.strip()
            if tag:
                nods_tags.append(tag)
nods_tag_counts = Counter(nods_tags)
top_gap_cats = nods_tag_counts.most_common(10)
detail = ' | '.join(f'{tag}: {cnt}' for tag, cnt in top_gap_cats)
qc_info('QC-09', f'BC_only_no_DS ({len(nods)} BCs) -- top categories', len(nods), detail)

# QC-10: Retired BCs in output
retired = bc_dss_df[
    bc_dss_df['BC_Name'].astype(str).str.upper().str.contains('RETIRED', na=False)
].drop_duplicates('COSMoS_BC_ID')
if len(retired) > 0:
    detail = ' | '.join(retired['BC_Name'].tolist()[:10])
    qc_flag('QC-10', 'WARNING', 'Retired BCs included in output', len(retired), detail)
else:
    qc_pass('QC-10', 'Retired BCs included in output')



# QC-11: Invalid Result_Scale values
# Valid per guidelines: Quantitative, Ordinal, Nominal, Narrative, Temporal
# Non-compliant values are COSMoS source data issues, not pipeline errors
valid_scales = {'Quantitative', 'Ordinal', 'Nominal', 'Narrative', 'Temporal'}
v_with_scale = v[~v['Result_Scale'].astype(str).str.strip().isin(['', 'nan'])]
invalid_scale = v_with_scale[~v_with_scale['Result_Scale'].isin(valid_scales)]
qualitative_rows = invalid_scale[invalid_scale['Result_Scale'] == 'Qualitative']
datetime_rows    = invalid_scale[invalid_scale['Result_Scale'] == 'datetime']
other_invalid    = invalid_scale[~invalid_scale['Result_Scale'].isin(['Qualitative', 'datetime'])]
# QC-11a: 'Qualitative' -- COSMoS systematic usage, not in curation principles valid set
if len(qualitative_rows) > 0:
    by_dom = qualitative_rows['Domain'].value_counts().to_dict()
    detail = ' | '.join(f'{d}: {n}' for d, n in sorted(by_dom.items()))
    qc_flag('QC-11a', 'WARNING',
            "Result_Scale='Qualitative' -- COSMoS source term, not in curation principles valid set",
            len(qualitative_rows), detail)
else:
    qc_pass('QC-11a', "Result_Scale='Qualitative' -- COSMoS source term check")
# QC-11b: 'datetime' -- likely maps to Temporal in curation principles
if len(datetime_rows) > 0:
    by_dom = datetime_rows['Domain'].value_counts().to_dict()
    detail = ' | '.join(f'{d}: {n}' for d, n in sorted(by_dom.items()))
    qc_flag('QC-11b', 'WARNING',
            "Result_Scale='datetime' -- likely maps to Temporal per curation principles",
            len(datetime_rows), detail)
else:
    qc_pass('QC-11b', "Result_Scale='datetime' -- Temporal mapping check")
# QC-11c: anything else invalid -- true ERROR
if len(other_invalid) > 0:
    by_val = other_invalid['Result_Scale'].value_counts().to_dict()
    detail = ' | '.join(f'{val}: {cnt}' for val, cnt in by_val.items())
    qc_flag('QC-11c', 'ERROR', 'Result_Scale: unexpected value (not Qualitative or datetime)', len(other_invalid), detail)
else:
    qc_pass('QC-11c', 'Result_Scale: no unexpected values')

# QC-12: Placeholder BC_IDs (NCIt code not yet assigned)
# Expected per process -- curators request NCIt codes for new concepts
# BC_ID should match C+digits; NEW_* indicates pending NCIt assignment
ncit_pat = re.compile(r'^C\d+$')
bc_unique = bc_dss_df.drop_duplicates('COSMoS_BC_ID')
placeholder_bcs = bc_unique[
    ~bc_unique['COSMoS_BC_ID'].astype(str).apply(lambda x: bool(ncit_pat.match(x.strip())))
]
if len(placeholder_bcs) > 0:
    detail = ' | '.join(
        f'{r["COSMoS_BC_ID"]} ({str(r["BC_Name"])[:30]})'
        for _, r in placeholder_bcs.iterrows()
    )
    qc_info('QC-12', 'Placeholder BC_IDs (NCIt code pending)', len(placeholder_bcs), detail)
else:
    qc_pass('QC-12', 'Placeholder BC_IDs (NCIt code pending)')

# QC-13: BC_Synonyms where preferred term not stripped
# PT should be removed from synonyms list (it is already in BC_Name)
bc_unique2 = bc_dss_df.drop_duplicates('COSMoS_BC_ID').copy()
has_syn = bc_unique2[~bc_unique2['BC_Synonyms'].astype(str).str.strip().isin(['', 'nan'])]
syn_first = has_syn['BC_Synonyms'].astype(str).apply(lambda x: x.split(' | ')[0].strip().lower())
bc_name_lower = has_syn['BC_Name'].astype(str).str.strip().str.lower()
syn_pt_present = has_syn[syn_first == bc_name_lower]
if len(syn_pt_present) > 0:
    detail = ' | '.join(
        f'{r["COSMoS_BC_ID"]} ({str(r["BC_Name"])[:30]})'
        for _, r in syn_pt_present.iterrows()
    )
    qc_flag('QC-13', 'WARNING', 'BC_Synonyms: preferred term not stripped', len(syn_pt_present), detail)
else:
    qc_pass('QC-13', 'BC_Synonyms: preferred term not stripped')

# QC-14: TESTCD_NCIt differs from NCIt_Code
# A small number of DSS rows carry a TESTCD assigned_term (NCIt code) that
# differs from the parent BC's NCIt_Code. Reason not confirmed -- possibly
# legacy pre-COSMoS NCIt assignments. Informational, not an error.
if 'TESTCD_NCIt' in bc_dss_df.columns:
    ncit_mismatch = v[
        (v['TESTCD_NCIt'].str.len() > 0)
        & (v['NCIt_Code'].str.len() > 0)
        & (v['TESTCD_NCIt'] != v['NCIt_Code'])
    ]
    if len(ncit_mismatch) > 0:
        detail = ' | '.join(
            f'{r["TESTCD"]} ({r["BC_Name"][:30]}): BC={r["NCIt_Code"]} TESTCD={r["TESTCD_NCIt"]}'
            for _, r in ncit_mismatch.iterrows()
        )
        qc_info('QC-14', 'TESTCD_NCIt differs from NCIt_Code', len(ncit_mismatch), detail)
    else:
        qc_pass('QC-14', 'TESTCD_NCIt differs from NCIt_Code')
else:
    qc_pass('QC-14', 'TESTCD_NCIt column not present -- check skipped')

# Summary
n_error   = sum(1 for r in qc_results if r['Severity'] == 'ERROR')
n_warning = sum(1 for r in qc_results if r['Severity'] == 'WARNING')
n_info    = sum(1 for r in qc_results if r['Severity'] == 'INFO')
n_pass    = sum(1 for r in qc_results if r['Severity'] == 'PASS')

print(f'\nQC Summary: {n_pass} PASS  {n_error} ERROR  {n_warning} WARNING  {n_info} INFO')
qc_df = pd.DataFrame(qc_results)
print(f'QC results captured: {len(qc_df)} checks')


## 6. Export QC Report

In [ ]:
README_HEADER  = PatternFill(start_color='595959', end_color='595959', fill_type='solid')
README_SECTION = PatternFill(start_color='D9D9D9', end_color='D9D9D9', fill_type='solid')
HEADER_FONT    = Font(bold=True)
HEADER_FONT_WHITE = Font(bold=True, color='FFFFFF')
thin_border = Border(
    left=Side(style='thin', color='999999'),
    right=Side(style='thin', color='999999'),
    top=Side(style='thin', color='999999'),
    bottom=Side(style='thin', color='999999'),
)

SEV_FILLS = {
    'PASS':    PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid'),
    'INFO':    PatternFill(start_color='FFFCE8', end_color='FFFCE8', fill_type='solid'),
    'WARNING': PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid'),
    'ERROR':   PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid'),
}
SEV_FONTS = {
    'PASS':    Font(color='276221'),
    'INFO':    Font(color='7B6000'),
    'WARNING': Font(bold=True, color='9C6500'),
    'ERROR':   Font(bold=True, color='9C0006'),
}

qc_df = pd.DataFrame(qc_results)

# README content for QC report
readme_data = [
    ['SECTION', 'ITEM', 'DESCRIPTION'],
    ['', '', ''],
    ['PURPOSE', '', 'QC report for COSMoS_BC_DSS_Flatten output'],
    ['', '', f'Validates interim/COSMoS_BC_DSS.xlsx against structural and curation-principle checks'],
    ['', '', f'Generated: {GENERATION_DATE}'],
    ['', '', ''],
    ['SHEETS', '', ''],
    ['', 'README', 'This sheet'],
    ['', 'QC_Report', 'All QC check results with severity, count, and detail'],
    ['', '', ''],
    ['CHECK TYPES', '', ''],
    ['', 'QC-01 to QC-10', 'Structural pipeline checks'],
    ['', 'QC-11 to QC-13', 'Validated against CDISC BC Curation Principles and Completion Guidelines'],
    ['', '', '(COSMoS bc_starter_package -- BC Curation Principles and Completion GLs.xlsx)'],
    ['', '', ''],
    ['SEVERITY', '', ''],
    ['', 'ERROR', 'Structural integrity failure -- investigate before using output'],
    ['', 'WARNING', 'Potential issue -- review recommended, may be expected for some COSMoS versions'],
    ['', 'INFO', 'Informational -- documents expected patterns or gap analysis'],
    ['', 'PASS', 'Check passed, no issues found'],
]

with pd.ExcelWriter(REPORT_FILE, engine='openpyxl') as writer:
    readme_df = pd.DataFrame(readme_data[1:], columns=readme_data[0])
    readme_df.to_excel(writer, sheet_name='README', index=False)
    qc_df.to_excel(writer, sheet_name='QC_Report', index=False)

# Format
wb = load_workbook(REPORT_FILE)

# README
ws = wb['README']
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=True, vertical='top')
for cell in ws[1]:
    cell.font = HEADER_FONT_WHITE
    cell.fill = README_HEADER
for row in ws.iter_rows(min_row=2):
    val = str(row[0].value) if row[0].value else ''
    if val and val == val.upper() and val.strip():
        for cell in row:
            cell.font = HEADER_FONT
            cell.fill = README_SECTION
ws.column_dimensions['A'].width = 20
ws.column_dimensions['B'].width = 20
ws.column_dimensions['C'].width = 80

# QC_Report
ws = wb['QC_Report']
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(wrap_text=True, vertical='top')
for cell in ws[1]:
    cell.font = HEADER_FONT_WHITE
    cell.fill = README_HEADER
ws.column_dimensions['A'].width = 10
ws.column_dimensions['B'].width = 12
ws.column_dimensions['C'].width = 55
ws.column_dimensions['D'].width = 8
ws.column_dimensions['E'].width = 80

sev_col_idx = [cell.value for cell in ws[1]].index('Severity') + 1
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    sev = row[sev_col_idx - 1].value or ''
    fill = SEV_FILLS.get(sev, PatternFill(start_color='FFFCE8', end_color='FFFCE8', fill_type='solid'))
    font = SEV_FONTS.get(sev, Font())
    for cell in row:
        cell.fill = fill
    row[sev_col_idx - 1].font = font

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

wb.save(REPORT_FILE)
print(f'[OK] QC report written to {REPORT_FILE.name}')

## 7. Final Summary

In [ ]:
print('=' * 70)
print('COSMoS_BC_DSS_Validate -- COMPLETE')
print('=' * 70)
print(f'\nInput:  {INTERIM_FILE.name}')
print(f'Output: {REPORT_FILE.name}')
print(f'\nQC Summary: {n_pass} PASS  {n_error} ERROR  {n_warning} WARNING  {n_info} INFO')
print(f'\nChecks:')
for r in qc_results:
    tag = f'[{r["Severity"]}]'
    print(f'  {tag:9} {r["Check"]} {r["Description"]} -- {r["Count"]}')
if n_error > 0:
    print(f'\n  *** {n_error} ERROR(s) found -- review QC_Report before downstream use ***')
print('=' * 70)